In [1]:
from sklearn.datasets import fetch_openml
import pandas as pd

data = fetch_openml('credit-g', version=1, as_frame=True)
X, y = data.data, data.target

print(f"Shape: {X.shape}")
print(f"\nClass counts:\n{y.value_counts()}")
print(f"\nClass proportions:\n{y.value_counts(normalize=True)}")

Shape: (1000, 20)

Class counts:
class
good    700
bad     300
Name: count, dtype: int64

Class proportions:
class
good    0.7
bad     0.3
Name: proportion, dtype: float64


In [2]:
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()

print(f"Numeric ({len(num_cols)}):\n{num_cols}\n")
print(f"Categorical ({len(cat_cols)}):\n{cat_cols}\n")
print(f"Missing values total: {X.isnull().sum().sum()}")

Numeric (7):
['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']

Categorical (13):
['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']

Missing values total: 0


In [3]:
banned = ['personal_status', 'foreign_worker']

X_full = X.copy()
X_clean = X.drop(columns=banned)

cat_cols_clean = [c for c in cat_cols if c not in banned]

print(f"Original: {X_full.shape[1]} columns")
print(f"Clean:    {X_clean.shape[1]} columns")
print(f"Removed:  {banned}")

Original: 20 columns
Clean:    18 columns
Removed:  ['personal_status', 'foreign_worker']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape[0]} rows")
print(f"Test:  {X_test.shape[0]} rows")
print(f"\nTrain mix:\n{y_train.value_counts(normalize=True)}")
print(f"Test mix:\n{y_test.value_counts(normalize=True)}")

Train: 800 rows
Test:  200 rows

Train mix:
class
good    0.7
bad     0.3
Name: proportion, dtype: float64
Test mix:
class
good    0.7
bad     0.3
Name: proportion, dtype: float64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols_clean = [c for c in num_cols if c not in banned]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_clean),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_clean)
    ]
)

print(f"Numeric columns to scale: {len(num_cols_clean)}")
print(f"Word columns to encode:   {len(cat_cols_clean)}")

Numeric columns to scale: 7
Word columns to encode:   11


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

logreg.fit(X_train, y_train)

print("Model trained.")
print(f"Learned from {X_train.shape[0]} people.")

Model trained.
Learned from 800 people.


In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba, labels=['bad', 'good'])

print(f"Accuracy: {acc:.3f}   (baseline to beat: 0.700)")
print(f"AUC:      {auc:.3f}")
print()
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.700   (baseline to beat: 0.700)
AUC:      0.747

[[ 30  30]
 [ 30 110]]

              precision    recall  f1-score   support

         bad       0.50      0.50      0.50        60
        good       0.79      0.79      0.79       140

    accuracy                           0.70       200
   macro avg       0.64      0.64      0.64       200
weighted avg       0.70      0.70      0.70       200



In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]

gb_acc = accuracy_score(y_test, gb_pred)
gb_auc = roc_auc_score(y_test, gb_proba, labels=['bad', 'good'])

print(f"{'Model':<22}{'Accuracy':<12}{'AUC'}")
print(f"{'Logistic Regression':<22}{acc:<12.3f}{auc:.3f}")
print(f"{'Gradient Boosting':<22}{gb_acc:<12.3f}{gb_auc:.3f}")
print()
print(confusion_matrix(y_test, gb_pred))

Model                 Accuracy    AUC
Logistic Regression   0.700       0.747
Gradient Boosting     0.740       0.772

[[ 31  29]
 [ 23 117]]


In [9]:
import numpy as np

COST_FALSE_APPROVE = 5000
COST_FALSE_REJECT  = 500

results = []
for t in np.arange(0.05, 0.96, 0.01):
    pred = np.where(gb_proba >= t, 'good', 'bad')
    cm = confusion_matrix(y_test, pred, labels=['bad', 'good'])
    false_approve = cm[0, 1]
    false_reject  = cm[1, 0]
    cost = false_approve * COST_FALSE_APPROVE + false_reject * COST_FALSE_REJECT
    results.append((t, false_approve, false_reject, cost))

best = min(results, key=lambda r: r[3])

print(f"{'Threshold':<12}{'Bad approved':<15}{'Good rejected':<16}{'Cost'}")
for t, fa, fr, c in results[::10]:
    print(f"{t:<12.2f}{fa:<15}{fr:<16}${c:,}")

print(f"\nBest threshold: {best[0]:.2f}")
print(f"  Defaulters approved: {best[1]}  (was 29 at 0.50)")
print(f"  Good customers lost: {best[2]}")
print(f"  Total cost: ${best[3]:,}")

Threshold   Bad approved   Good rejected   Cost
0.05        55             0               $275,000
0.15        46             6               $233,000
0.25        38             10              $195,000
0.35        35             16              $183,000
0.45        31             23              $166,500
0.55        25             24              $137,000
0.65        19             33              $111,500
0.75        16             45              $102,500
0.85        14             63              $101,500
0.95        5              86              $68,000

Best threshold: 0.94
  Defaulters approved: 5  (was 29 at 0.50)
  Good customers lost: 83
  Total cost: $66,500


In [10]:
MIN_APPROVAL_RATE = 0.70

practical = [r for r in results
             if (r[1] + (140 - r[2])) / 200 >= MIN_APPROVAL_RATE]

best_practical = min(practical, key=lambda r: r[3])

print(f"Unconstrained best: {best[0]:.2f}  cost ${best[3]:,}  "
      f"approval rate {(best[1] + (140 - best[2]))/200:.0%}")
print(f"Constrained best:   {best_practical[0]:.2f}  cost ${best_practical[3]:,}  "
      f"approval rate {(best_practical[1] + (140 - best_practical[2]))/200:.0%}")
print(f"\nAt {best_practical[0]:.2f}: {best_practical[1]} defaulters approved, "
      f"{best_practical[2]} good customers rejected")

Unconstrained best: 0.94  cost $66,500  approval rate 31%
Constrained best:   0.54  cost $136,500  approval rate 71%

At 0.54: 25 defaulters approved, 23 good customers rejected


In [11]:
logreg.fit(X_train, y_train)

feature_names = logreg.named_steps['prep'].get_feature_names_out()
coefs = logreg.named_steps['model'].coef_[0]

importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs
}).sort_values('coefficient')

print("STRONGEST DEFAULT SIGNALS (push toward bad credit)")
print(importance.head(10).to_string(index=False))
print()
print("STRONGEST REPAYMENT SIGNALS (push toward good credit)")
print(importance.tail(10).to_string(index=False))

STRONGEST DEFAULT SIGNALS (push toward bad credit)
                                  feature  coefficient
                   cat__purpose_education    -0.759101
                  cat__checking_status_<0    -0.716000
                     cat__purpose_new car    -0.685322
                 cat__savings_status_<100    -0.612126
          cat__other_parties_co applicant    -0.526602
cat__property_magnitude_no known property    -0.524679
            cat__checking_status_0<=X<200    -0.410658
                        cat__housing_rent    -0.406037
                     cat__purpose_repairs    -0.395918
             cat__credit_history_all paid    -0.384333

STRONGEST REPAYMENT SIGNALS (push toward good credit)
                                           feature  coefficient
                             cat__housing_for free     0.323627
                             cat__purpose_radio/tv     0.345842
              cat__savings_status_no known savings     0.347188
                     cat__other_p

In [12]:
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)

prep_full = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

gb_full = Pipeline(steps=[
    ('prep', prep_full),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

gb_full.fit(Xf_train, yf_train)
full_proba = gb_full.predict_proba(Xf_test)[:, 1]
full_auc = roc_auc_score(yf_test, full_proba, labels=['bad', 'good'])

print(f"{'Feature set':<32}{'Columns':<10}{'AUC'}")
print(f"{'All features (incl. protected)':<32}{X_full.shape[1]:<10}{full_auc:.3f}")
print(f"{'Legally compliant':<32}{X_clean.shape[1]:<10}{gb_auc:.3f}")
print(f"\nCost of compliance: {full_auc - gb_auc:+.3f} AUC")

Feature set                     Columns   AUC
All features (incl. protected)  20        0.764
Legally compliant               18        0.772

Cost of compliance: -0.007 AUC
